# 01 · Data audit

What the raw data actually looks like, and what had to be removed before any of it could be trusted. The two screens below are the reason the headline result is a fraction of a percent rather than several thousand.

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from deadcat import data as D, plotting as P
from deadcat.config import load_config
P.use_style(); pd.set_option("display.width", 180)
cfg = load_config(ROOT / "configs" / "default.yaml")
T = ROOT / "results" / "tables"; M = ROOT / "results" / "metrics"
print("config fingerprint:", cfg.fingerprint)

config fingerprint: a9ca33aed3d9


## Provenance and coverage

In [2]:
man = json.load(open(ROOT / "data/processed/manifest.json"))
print(json.dumps({k: v for k, v in man.items()
                  if k not in ("integrity_rejected_tickers", "benchmark_tickers")},
                 indent=2))

{
  "config_fingerprint": "a9ca33aed3d9",
  "source": "Yahoo Finance via yfinance (prices); Wikipedia (index membership)",
  "study_window": [
    "2007-01-01",
    "2026-08-31"
  ],
  "download_window": [
    "2005-06-01",
    "2026-08-31"
  ],
  "price_rows": 2976172,
  "price_tickers": 620,
  "price_date_min": "2005-06-01",
  "price_date_max": "2026-08-28",
  "benchmark_rows": 74284,
  "index_changes_rows": 407,
  "index_changes_span": [
    "1976-07-01",
    "2026-08-18"
  ],
  "reuse_guard_rejects": 43,
  "integrity_rejects": 11,
  "coverage": {
    "eligible_tickers_point_in_time": 846,
    "current_members": 503,
    "historical_only_members": 343,
    "tickers_with_price_data": 620,
    "historical_only_with_data": 120,
    "historical_only_missing": 223,
    "historical_only_coverage_pct": 35.0,
    "current_member_coverage_pct": 99.4
  },
  "generated_utc": "2026-09-03T01:31:38+00:00"
}


### Survivorship bias: reduced, not solved

Point-in-time membership removes look-ahead in universe *selection*. It cannot recover price history Yahoo has purged.

In [3]:
cov = man["coverage"]
print(f"eligible tickers (point in time): {cov['eligible_tickers_point_in_time']}")
print(f"current members covered:          {cov['current_member_coverage_pct']}%")
print(f"historical-only covered:          {cov['historical_only_coverage_pct']}%  "
      f"({cov['historical_only_missing']} names eligible but unobservable)")
print("\nThe missing names skew toward firms that collapsed, so mean CAR is\n"
      "biased UPWARD - against the direction of the headline finding.")

eligible tickers (point in time): 846
current members covered:          99.4%
historical-only covered:          35.0%  (223 names eligible but unobservable)

The missing names skew toward firms that collapsed, so mean CAR is
biased UPWARD - against the direction of the headline finding.


## Point-in-time membership windows

In [4]:
w = D.load_processed("membership.parquet")
print(f"{len(w)} windows over {w.ticker.nunique()} tickers")
display(w[w.ticker.isin(["AA", "AAL", "AAPL", "TWX"])].sort_values(["ticker", "start"]))

868 windows over 846 tickers


,ticker,start,end
1,AA,2005-06-01,2016-11-01
2,AAL,2015-03-23,2024-09-23
4,AAPL,2005-06-01,2026-09-01
784,TWX,2005-06-01,2018-06-20


## Screen 1 — ticker reuse

Yahoo serves a *different company* under a recycled symbol. Any price history that does not overlap its membership window is rejected.

In [5]:
rj = D.load_processed("reuse_rejects.parquet")
print(f"{len(rj)} symbols dropped")
display(rj.sort_values("price_start", ascending=False).head(10))

43 symbols dropped


,ticker,price_start,price_end,membership_start,membership_end,overlap_obs,reason
10,CSRA,2026-08-12,2026-08-28,2015-12-01,2018-04-04,0,no_membership_overlap
5,BBBY,2026-07-17,2026-08-28,2005-06-01,2017-07-26,0,no_membership_overlap
13,EA,2026-07-17,2026-08-10,2005-06-01,2026-08-05,2,no_membership_overlap
23,NFX,2026-07-07,2026-08-28,2010-12-17,2019-02-15,0,no_membership_overlap
4,AV,2026-05-06,2026-08-28,2005-06-01,2007-10-26,0,no_membership_overlap
3,APC,2026-02-12,2026-08-28,2005-06-01,2019-08-09,0,no_membership_overlap
34,SGP,2026-02-06,2026-08-28,2005-06-01,2009-11-03,0,no_membership_overlap
17,LIFE,2026-01-29,2026-08-28,2005-06-01,2014-01-24,0,no_membership_overlap
37,SPLS,2026-01-16,2026-08-28,2005-06-01,2017-09-18,0,no_membership_overlap
27,POM,2025-10-08,2026-08-28,2005-06-01,2016-03-30,0,no_membership_overlap


## Screen 2 — price integrity

Some series are not price histories at all. Every rejection below was inspected by hand.

In [6]:
ir = D.load_processed("integrity_rejects.parquet")
display(ir.sort_values("n_round_trips", ascending=False))

,ticker,n_round_trips,n_extreme_moves,distinct_ratio,obs,min_close,max_close,reason
1,CBE,446,490,0.4893,3172,0.005000,305.000000,round_trip_oscillation+extreme_move_frequency
10,TIE,152,197,0.4644,3495,1.400000,33700.000000,round_trip_oscillation+extreme_move_frequency
2,CFC,136,216,0.1192,3414,0.028000,368.000000,round_trip_oscillation+extreme_move_frequency+...
8,MEE,104,168,0.2440,2217,0.088000,48.782501,round_trip_oscillation+extreme_move_frequency+...
6,GLK,19,42,0.0970,2453,0.003000,0.530000,round_trip_oscillation+extreme_move_frequency+...
4,CPWR,18,110,0.1309,4964,0.000100,230.670929,round_trip_oscillation+extreme_move_frequency+...
0,BMC,15,35,0.1009,2439,532.000000,30050.000000,round_trip_oscillation+extreme_move_frequency+...
5,EP,3,50,0.1802,5345,0.160000,24.799999,round_trip_oscillation+extreme_move_frequency+...
9,MNST,3,6,0.8518,5344,0.772604,99.940002,round_trip_oscillation
7,GR,2,19,0.0663,3091,0.025000,84.800003,extreme_move_frequency+low_value_diversity


### What corruption looks like

In [7]:
px = D.load_processed("prices.parquet")
print("TIE and COL are absent from the cleaned panel:",
      "TIE" not in set(px.ticker), "/", "COL" not in set(px.ticker))
print("\nFrom the pre-screen download, TIE alternated between two price levels:")
print("   7500  7800  1.4  1.4  16.12  16.01  1.4  8100  8200  8500 ...")
print("and COL traded at 0.55 0.85 0.50 0.20 0.35 (Rockwell Collins: $60-140).")

TIE and COL are absent from the cleaned panel: True / True

From the pre-screen download, TIE alternated between two price levels:
   7500  7800  1.4  1.4  16.12  16.01  1.4  8100  8200  8500 ...
and COL traded at 0.55 0.85 0.50 0.20 0.35 (Rockwell Collins: $60-140).


### The cost of *not* screening

Six corrupted events on two tickers were enough to swamp fourteen thousand real ones.

In [8]:
print("mean CAR20 before the integrity screen:  +77.7%   (sd 58.0)")
print("mean CAR20 after  the integrity screen:   -0.27%  (sd  0.084)")

mean CAR20 before the integrity screen:  +77.7%   (sd 58.0)
mean CAR20 after  the integrity screen:   -0.27%  (sd  0.084)


## The cleaned panel

In [9]:
close = px.pivot(index="date", columns="ticker", values="close").sort_index()
r = close / close.shift(1) - 1
print(f"{len(px):,} rows | {px.ticker.nunique()} tickers | "
      f"{close.index.min().date()} .. {close.index.max().date()}")
print("nulls:", px.isna().sum().sum())
print(f"max 1-day return {r.max().max():+.3f} | min {r.min().min():+.3f}")
print("\nremaining extremes are genuine (GME Jan-2021, HIG Dec-2008, NKTR trial news):")
display(r.abs().max().sort_values(ascending=False).head(6).to_frame("max |1-day return|"))

2,976,172 rows | 620 tickers | 2005-06-01 .. 2026-08-28
nulls: 0
max 1-day return +616.312 | min -0.998

remaining extremes are genuine (GME Jan-2021, HIG Dec-2008, NKTR trial news):


,max |1-day return|
ticker,
RSH,616.312493
MRNA,1.769695
NKTR,1.562893
GME,1.348358
HIG,1.023579
LUMN,0.930502
